# Bronze Quality Profiling — Etapa 2 (itens 2 e 3)

Gera as evidências dos itens **2. Dicionário de dados** e **3. Qualidade dos dados**.

Roda sobre as tabelas **bronze**, e não sobre o silver, porque o `bronze_to_silver`
converte os cinco códigos de ausência (`Recusa`, `Não informou`, `Não sabe informar`,
`Não sabe`, `-`) em um único NULL. Depois dessa conversão não é mais possível medir
*que tipo* de ausência cada município produziu — que é justamente a evidência pedida.

Este notebook é **somente leitura sobre os dados**: não altera nenhuma tabela de
bronze, silver ou gold do produto — grava apenas os artefatos de evidência `dq_*`
(com o widget `persistir_tabelas` ligado) e CSVs de apoio.

**Atenção — as `dq_*` não são só evidência, são insumo da gold (Etapa 3):** o
`silver_to_gold` constrói a `dim_indicador` **diretamente** a partir de
`dq_column_profile` (significados e tipos) e `dq_skiplogic_candidates` (pais de
salto lógico). Rodar com `persistir_tabelas = "nao"` é legítimo para prévia
somente leitura, mas deixa a gold com a evidência gravada pela última execução
com `"sim"` da edição corrente — e, num workspace novo, o `silver_to_gold` aborta
até existir ao menos uma execução com `"sim"` para a edição.

**Saídas** — todas gravadas no schema `gold` com o prefixo `dq_`, separando artefatos de qualidade do produto analítico pelo nome, sem criar um schema novo.

**`dq_column_profile`** — perfil de qualidade por coluna

* **Granularidade:** uma linha por coluna das duas tabelas de dados da bronze.
* **Conteúdo:** tipo detectado (binário / categórico / numérico / texto livre / sem resposta), domínio de valores observado, contagem e percentual de cada classe de ausência, significado e sua origem (dicionário IBGE / inferido pelo grupo / não documentado) e status no funil de seleção.
* **Para quê:** é o esqueleto do item 2 (dicionário de dados) do documento.

**`dq_dimension_summary`** — síntese das dimensões de qualidade

* **Granularidade:** uma linha por dimensão — as seis: completude, validade, consistência, unicidade, atualidade e acurácia.
* **Conteúdo:** o que foi verificado, o número medido e o veredito.
* **Para quê:** é a tabela 3.7 do documento de entrega da parte 2, pronta para colar.

**`dq_nonresponse_by_size`** — viés por porte do município

* **Granularidade:** uma linha por faixa populacional.
* **Conteúdo:** não-resposta no bloco de LGPD e a taxa de "Sim" da variável-alvo.
* **Para quê:** evidência de que a ausência não é aleatória — municípios pequenos respondem menos *e* respondem "Sim" menos.

**`dq_skiplogic_candidates`** — candidatos a filtro de salto

* **Granularidade:** uma linha por par de colunas pai/filha.
* **Conteúdo:** o conjunto de respostas que dispara o salto, a cobertura e a precisão da relação, o veredito estatístico (`veredito_dados`) e a verificação manual contra o questionário (`verificacao_questionario`, NULL = pendente).
* **Para quê:** hipóteses a confirmar contra o questionário (dimensão acurácia) — são a base dos filtros usados pelo gold.

**`dq_dict_drift`** — divergência do dicionário entre edições

* **Granularidade:** uma linha por código comum a duas edições cujo significado divergiu.
* **Conteúdo:** o significado em cada edição e o tipo de divergência (reescrita editorial, ano de referência embutido, reformulação parcial, suspeita de reutilização do código).
* **Para quê:** aviso a quem escreve a configuração do radar de uma edição — um código não significa a mesma coisa em todos os anos.

**Edições:** cada `dq_*` acima, **exceto a `dq_dict_drift`**, acumula todas as edições já perfiladas numa única tabela — a coluna `ano_edicao` diferencia as execuções e cada gravação substitui apenas os dados da edição corrente, preservando as anteriores. A `dq_dict_drift` compara os anos **entre si** (colunas `edicao_a`/`edicao_b`) e por isso é regravada inteira a cada execução.

Nada é gravado enquanto o widget `persistir_tabelas` estiver em `nao`.

## Visão geral — as etapas deste notebook, na ordem

1. **Parâmetros (widgets)** — gravar ou não as saídas no catálogo e os dois cortes do
   funil de seleção (não-resposta e concentração).
2. **Carga** — as duas tabelas de dados e o dicionário da bronze, **apenas da edição
   corrente** (`reference_year` das variáveis de ambiente).
3. **Formato longo** — cada resposta vira uma linha (município × pergunta × valor),
   classificada em uma de seis classes (preenchida, não sabe, não informou, recusa,
   não aplicável, nula).
4. **Item 2 — perfil por coluna** — tipo, domínio, ausências e significado de cada
   coluna de indicador (dicionário IBGE, inferido pelo grupo ou não documentado).
   Depois, o **2.1**: divergência de significado dos códigos entre **todas** as
   edições carregadas.
5. **Item 3 — qualidade em seis dimensões**:
   * **3.1 Completude** — das perguntas aplicáveis, quantas foram respondidas;
   * **3.2 Validade** — os valores respeitam o domínio esperado da coluna;
   * **3.3 Unicidade** — a chave `cod_munic` não se repete;
   * **3.4 Consistência** — descritores do município idênticos nas duas abas;
   * **3.5 Atualidade** — defasagem entre ano de referência, ingestão e uso;
   * **3.6 Acurácia** — coerência entre perguntas encadeadas (filtros de salto).

   Mais a evidência de viés: municípios pequenos respondem menos e dizem "Sim" menos.
6. **Funil de seleção** — critérios explícitos registram quais colunas sobrevivem
   ao recorte analítico; **o funil só registra — não remove coluna de nenhuma
   tabela** (o recorte de fato é aplicado no gold, pela configuração do radar).
7. **3.7 Síntese** — uma linha por dimensão: verificação, evidência e avaliação
   (tabela 3.7 do documento).
8. **Gráficos** — as três figuras do documento.
9. **Persistência** — com `persistir_tabelas = sim`: `dq_*` no gold, gravadas **por
   edição** (as anteriores são preservadas), e CSVs no volume de landing.

```text
bronze — edição corrente (reference_year)
   │  formato longo + classificação da ausência (6 classes)
   ▼
perfil por coluna ──► item 2: dicionário IBGE ──► 2.1: divergência entre edições
   │
   ▼
item 3: 3.1 completude · 3.2 validade · 3.3 unicidade
        3.4 consistência · 3.5 atualidade · 3.6 filtros de salto
        └─ viés por porte: municípios pequenos respondem (e dizem "Sim") menos
   ▼
funil de seleção ──► 3.7 síntese ──► gráficos
   │  persistir_tabelas = sim
   ▼
gold.dq_* (por edição, preservando as anteriores) + CSVs no volume
```

In [0]:
%run ../environment/env_functions

In [0]:
# Parâmetros do notebook. No Databricks, aparecem como caixas de entrada no topo da
# página. Rodando FORA do Databricks (Jupyter local, testes), o dbutils não existe —
# o `except NameError` abaixo entra e usa os VALORES_PADRAO definidos aqui, que podem
# ser editados diretamente neste bloco. Os padrões são seguros: nada é gravado no
# Unity Catalog até persistir_tabelas = "sim".
VALORES_PADRAO = {
    "persistir_tabelas": "sim",
    "limiar_nao_resposta": "50",
    "limiar_concentracao": "97",
}

try:
    dbutils.widgets.dropdown("persistir_tabelas", VALORES_PADRAO["persistir_tabelas"],
                             ["sim", "nao"], "Gravar tabelas no catálogo?")
    dbutils.widgets.text("limiar_nao_resposta", VALORES_PADRAO["limiar_nao_resposta"],
                         "Descartar acima de X% de não-resposta")
    dbutils.widgets.text("limiar_concentracao", VALORES_PADRAO["limiar_concentracao"],
                         "Descartar acima de X% em um único valor")

    # Lê o widget defensivamente: o valor é o que o widget tinha quando ESTA célula
    # rodou — mudar o dropdown na interface não faz nada até a célula rodar de novo.
    _persistir_raw = dbutils.widgets.get("persistir_tabelas")
    _limiar_nr_raw = dbutils.widgets.get("limiar_nao_resposta")
    _limiar_conc_raw = dbutils.widgets.get("limiar_concentracao")
    origem_parametros = "widgets do Databricks"
except NameError:
    # Sem Databricks: os valores padrão mutáveis do código entram no lugar.
    _persistir_raw = VALORES_PADRAO["persistir_tabelas"]
    _limiar_nr_raw = VALORES_PADRAO["limiar_nao_resposta"]
    _limiar_conc_raw = VALORES_PADRAO["limiar_concentracao"]
    origem_parametros = "valores padrão do código (sem dbutils)"

PERSISTIR = _persistir_raw.strip().lower() in ("sim", "s", "true", "yes", "1")

LIMIAR_NAO_RESPOSTA = float(_limiar_nr_raw)
LIMIAR_CONCENTRACAO = float(_limiar_conc_raw)

# Imprimir a string bruta deixa isso óbvio, em vez de se comportar em silêncio como "nao".
print(f"Origem dos parâmetros  : {origem_parametros}")
print(f"Valor bruto do widget : {_persistir_raw!r}")
print(f"Gravar tabelas         : {PERSISTIR}")
print(f"Corte de não-resposta  : {LIMIAR_NAO_RESPOSTA}%")
print(f"Corte de concentração  : {LIMIAR_CONCENTRACAO}%")

In [0]:
%run ../environment/env_variables

In [0]:
# Bibliotecas usadas em todo o notebook
import pyspark.sql.functions as F
from pyspark.sql.window import Window

import pandas as pd
import numpy as np

In [0]:
# Tabelas de origem (camada bronze, gravadas pelo landing_to_bronze)
bronze_tbl_dict = f'{catalog_name}.{bronze_schema}.munic_raw_dict'
bronze_tbl_info = f'{catalog_name}.{bronze_schema}.munic_raw_info_data'
bronze_tbl_gov = f'{catalog_name}.{bronze_schema}.munic_raw_gov_data'

# Tabelas de saída. Vivem no gold com o prefixo dq_ para que os artefatos de
# qualidade fiquem separados do produto analítico pelo nome, sem criar um schema
# novo (o que exigiria permissão de CREATE SCHEMA).
dq_tbl_profile = f'{catalog_name}.{gold_schema}.dq_column_profile'
dq_tbl_summary = f'{catalog_name}.{gold_schema}.dq_dimension_summary'
dq_tbl_bysize = f'{catalog_name}.{gold_schema}.dq_nonresponse_by_size'
dq_tbl_skip = f'{catalog_name}.{gold_schema}.dq_skiplogic_candidates'
dq_tbl_drift = f'{catalog_name}.{gold_schema}.dq_dict_drift'

# Os cinco códigos de ausência usados pelo IBGE, mantidos SEPARADOS de propósito.
# O "-" é estruturalmente diferente dos outros quatro: significa que a pergunta não
# se aplicava àquele município (roteiro de salto do questionário), e não que o
# município deixou de responder. Colapsá-los é uma decisão de modelagem da silver;
# medi-los separados é o objetivo deste notebook.
COD_NAO_SABE = ["Não sabe informar", "Não sabe"]
COD_NAO_INFORMOU = ["Não informou"]
COD_RECUSA = ["Recusa"]
COD_NAO_APLICAVEL = ["-"]

# Colunas de identificação compartilhadas pelas duas abas; não são indicadores da
# pesquisa e precisam ficar de fora do perfilamento por indicador.
COLS_IDENTIFICACAO = [
    "cod_munic", "uf", "cod_uf", "desc_mun",
    "populacao", "faixa_populacao", "regiao",
]
# Colunas adicionadas pela ingestão da bronze (linhagem, não conteúdo da pesquisa).
COLS_METADADOS = [
    "source_file", "source_url", "dataset_name",
    "reference_year", "ingestion_timestamp",
    "ano_edicao",   # coluna de partição adicionada pelo upsert_delta_table: não é indicador
]

## Parte 1 — Inventário das tabelas bronze

Contagens básicas que alimentam o item 1 do documento e servem de linha de base
para as verificações seguintes.

In [0]:
# Filtra pela edição corrente (reference_year, vindo do env_variables): a bronze é particionada
# por ano_edicao e, com mais de uma edição, cada município apareceria repetido no df_long
# (o pivot do detector de salto lógico falharia com "Index contains duplicate entries").
df_raw_info = spark.table(bronze_tbl_info).filter(F.col("ano_edicao") == reference_year)
df_raw_gov = spark.table(bronze_tbl_gov).filter(F.col("ano_edicao") == reference_year)
df_raw_dict = spark.table(bronze_tbl_dict).filter(F.col("ano_edicao") == reference_year)

# Colunas de indicador = tudo que não é identificação nem metadado de ingestão
cols_ind_info = [c for c in df_raw_info.columns
                 if c not in COLS_IDENTIFICACAO + COLS_METADADOS]
cols_ind_gov = [c for c in df_raw_gov.columns
                if c not in COLS_IDENTIFICACAO + COLS_METADADOS]

# Falha ruidosamente se o schema da bronze mudou de um jeito que quebra a suposição de prefixo
assert cols_ind_info, "Nenhuma coluna de indicador encontrada na tabela info"
assert cols_ind_gov, "Nenhuma coluna de indicador encontrada na tabela gov"

n_info = df_raw_info.count()
n_gov = df_raw_gov.count()

print(f"munic_raw_info_data: {n_info:,} linhas | {len(df_raw_info.columns)} colunas "
      f"({len(cols_ind_info)} indicadores)")
print(f"munic_raw_gov_data : {n_gov:,} linhas | {len(df_raw_gov.columns)} colunas "
      f"({len(cols_ind_gov)} indicadores)")
print(f"munic_raw_dict     : {df_raw_dict.count():,} linhas")

## Parte 2 — Formato longo e classificação da ausência

Em vez de percorrer as centenas de colunas de dados uma a uma, as duas tabelas
são convertidas para formato longo — uma linha por (município, coluna, valor). Com isso, todas as
métricas das seis dimensões saem de agregações sobre um único DataFrame.

A função `classificar_ausencia` é a **única** definição de "o que conta como ausente"
no projeto: o mesmo critério alimenta o relatório de qualidade e a justificativa do
tratamento aplicado no silver.

In [0]:
def to_long(df, value_cols, tabela_nome):
    """
    Transforma um DataFrame largo em (tabela, cod_munic, coluna, valor).

    Usa o stack() do Spark para que o reshape inteiro seja uma única projeção, em
    vez de um SELECT por coluna. Todos os valores viram string porque as abas
    misturam colunas de texto e numéricas, e o que importa é a resposta bruta.

    Parâmetros:
        df (DataFrame)   : tabela da bronze
        value_cols (list): nomes das colunas de indicador a despivotar
        tabela_nome (str): rótulo mantido na saída para que as duas abas possam
                           compartilhar um único DataFrame longo

    Retorna:
        DataFrame com colunas: tabela, cod_munic, coluna, valor
    """
    # stack(n, 'col1', CAST(col1 AS STRING), 'col2', CAST(col2 AS STRING), ...)
    pares = ", ".join([f"'{c}', CAST(`{c}` AS STRING)" for c in value_cols])
    stack_expr = f"stack({len(value_cols)}, {pares}) as (coluna, valor)"

    return (
        df.select(
            F.lit(tabela_nome).alias("tabela"),
            F.col("cod_munic"),
            F.expr(stack_expr),
        )
    )


df_long_info = to_long(df_raw_info, cols_ind_info, "informatica_comunicacao")
df_long_gov = to_long(df_raw_gov, cols_ind_gov, "governanca")
df_long = df_long_info.unionByName(df_long_gov)

print(f"Formato longo: {len(cols_ind_info) + len(cols_ind_gov)} colunas "
      f"x {n_info:,} municípios")

In [0]:
display(df_long)

In [0]:
def classificar_ausencia(col_valor):
    """
    Classifica uma resposta bruta da pesquisa em uma de seis classes exclusivas.

    preenchido     -> uma resposta de fato
    nao_sabe       -> respondente não sabe ("Não sabe informar" / "Não sabe")
    nao_informou   -> respondente não forneceu a informação
    recusa         -> respondente se recusou a responder
    nao_aplicavel  -> "-", pergunta pulada pelo roteiro do questionário
    nulo           -> NULL ou string vazia no arquivo de origem

    As últimas cinco são todas "ausência", mas NÃO são a mesma coisa — e o
    relatório precisa distingui-las. Retorna uma expressão de Column, não um valor.
    """
    valor = F.trim(col_valor)
    return (
        F.when(col_valor.isNull(), "nulo")
        .when(valor == "", "nulo")
        .when(valor.isin(COD_NAO_SABE), "nao_sabe")
        .when(valor.isin(COD_NAO_INFORMOU), "nao_informou")
        .when(valor.isin(COD_RECUSA), "recusa")
        .when(valor.isin(COD_NAO_APLICAVEL), "nao_aplicavel")
        .otherwise("preenchido")
    )


df_long = df_long.withColumn("classe", classificar_ausencia(F.col("valor")))

# Sem .cache() aqui de propósito: o compute serverless rejeita PERSIST/CACHE
# (NOT_SUPPORTED_WITH_SERVERLESS). O reshape com stack() é barato o suficiente para
# recomputar, e o serverless aplica seu próprio cache em disco por baixo.

display(
    df_long.groupBy("classe").count()
    .withColumn("pct", F.round(F.col("count") / F.sum("count").over(Window.partitionBy()) * 100, 2))
    .orderBy(F.desc("count"))
)

## Item 2 — Dicionário de dados

Gera o esqueleto da tabela do item 2: **Campo**, **Tipo**, **Domínio** e **Qualidade**
saem do perfilamento; **Significado** vem da tabela de dicionário do bronze, quando o
código existe lá.

In [0]:
# Uma única agregação sobre o DataFrame longo produz todas as métricas por coluna
# de uma vez. O F.collect_set ignora NULLs, então envolver o valor em F.when(...)
# mantém apenas as respostas de fato preenchidas.
df_profile = (
    df_long.groupBy("tabela", "coluna")
    .agg(
        F.count("*").alias("n_registros"),
        F.sum(F.when(F.col("classe") == "preenchido", 1).otherwise(0)).alias("n_preenchido"),
        F.sum(F.when(F.col("classe") == "nao_sabe", 1).otherwise(0)).alias("n_nao_sabe"),
        F.sum(F.when(F.col("classe") == "nao_informou", 1).otherwise(0)).alias("n_nao_informou"),
        F.sum(F.when(F.col("classe") == "recusa", 1).otherwise(0)).alias("n_recusa"),
        F.sum(F.when(F.col("classe") == "nao_aplicavel", 1).otherwise(0)).alias("n_nao_aplicavel"),
        F.sum(F.when(F.col("classe") == "nulo", 1).otherwise(0)).alias("n_nulo"),
        F.collect_set(
            F.when(F.col("classe") == "preenchido", F.col("valor"))
        ).alias("valores_distintos"),
    )
)

# Colunas derivadas: percentuais, tipo detectado e uma amostra legível do domínio.
df_profile = (
    df_profile
    # Não-resposta = o respondente podia responder e não respondeu.
    # "nao_aplicavel" fica FORA desta soma de propósito: a pergunta nunca se aplicou.
    .withColumn("n_nao_resposta",
                F.col("n_nao_sabe") + F.col("n_nao_informou") + F.col("n_recusa") + F.col("n_nulo"))
    .withColumn("pct_preenchido", F.round(F.col("n_preenchido") / F.col("n_registros") * 100, 2))
    .withColumn("pct_nao_resposta", F.round(F.col("n_nao_resposta") / F.col("n_registros") * 100, 2))
    .withColumn("pct_nao_aplicavel", F.round(F.col("n_nao_aplicavel") / F.col("n_registros") * 100, 2))
    .withColumn("n_distintos", F.size("valores_distintos"))
    # Detecção de tipo, no mesmo espírito do treat_survey_indicators do bronze_to_silver:
    # forall() testa se TODOS os valores distintos são Sim ou Não.
    .withColumn(
        "tipo_detectado",
        F.when(F.col("n_distintos") == 0, "sem_resposta")
        .when(F.expr("forall(valores_distintos, x -> x in ('Sim', 'Não'))"), "binario")
        .when(F.col("n_distintos") <= 20, "categorico")
        .otherwise("texto_livre"),
    )
    # Amostra do domínio: até 6 valores distintos, em ordem alfabética, separados por "|".
    # Colunas de texto livre estourariam a célula, então recebem um rótulo.
    .withColumn(
        "dominio_amostra",
        F.when(F.col("tipo_detectado") == "texto_livre",
               F.concat(F.lit("texto livre ("), F.col("n_distintos").cast("string"), F.lit(" valores distintos)")))
        .otherwise(F.concat_ws(" | ", F.slice(F.array_sort("valores_distintos"), 1, 6))),
    )
)

In [0]:
# Uma coluna em que 99% dos municípios dão a mesma resposta carrega quase nenhuma
# informação para o agrupamento, mesmo 100% completa. Este é o segundo filtro do
# funil de seleção.
w_col = Window.partitionBy("tabela", "coluna").orderBy(F.desc("n"), F.asc("valor"))

df_concentracao = (
    df_long.filter(F.col("classe") == "preenchido")
    .groupBy("tabela", "coluna", "valor").agg(F.count("*").alias("n"))
    .withColumn("rk", F.row_number().over(w_col))
    .filter(F.col("rk") == 1)
    .select(
        "tabela", "coluna",
        F.col("valor").alias("valor_mais_frequente"),
        F.col("n").alias("n_valor_mais_frequente"),
    )
)

df_profile = (
    df_profile.join(df_concentracao, on=["tabela", "coluna"], how="left")
    .withColumn(
        "pct_concentracao",
        F.round(F.col("n_valor_mais_frequente") / F.col("n_preenchido") * 100, 2),
    )
)

In [0]:
# O dicionário guarda os códigos em maiúsculas (MGOV32) enquanto a bronze converteu
# os nomes de coluna para minúsculas (mgov32). Fazer o join por lower(cod_variable)
# é o que concilia os dois — e o fato de isso ser necessário já é, por si só, um
# achado para a seção de documentação.
df_dict_lookup = (
    df_raw_dict
    .select(F.lower(F.col("cod_variable")).alias("coluna"), F.col("meaning").alias("significado"))
    .dropDuplicates(["coluna"])
)

df_profile = df_profile.join(df_dict_lookup, on="coluna", how="left")

# Algumas colunas existem nos dados sem entrada na aba de dicionário do IBGE (a
# contagem varia por edição — o print abaixo mostra o número exato da corrente).
# Para as principais delas, o significado foi reconstruído a partir dos valores
# observados e da pergunta pai, e cada reconstrução fica registrada aqui para que a
# documentação possa afirmar exatamente quais definições são do grupo e quais vieram
# do IBGE.
SIGNIFICADOS_MANUAIS = {
    "mtic11": "Endereço eletrônico (URL) da página da prefeitura na internet",
    "mtic31": "Denominação da estrutura organizacional da área de comunicação",
    "mtic331": "Denominação do cargo ou carreira específica de comunicação social",
    "mtic2421": "Abrangência da conexão Wi-Fi na área urbana (parte ou toda)",
    "mtic2431": "Abrangência da conexão Wi-Fi na área rural (parte ou toda)",
    "mgov141": "Total de pessoal ocupado especificamente no controle interno (quantidade informada)",
    "mgov142": "Não sabe informar o total de pessoal ocupado no controle interno",
}

# Quantas colunas o dicionário do IBGE não documentou (evidência para o item 2.1),
# medida ANTES de os significados manuais serem aplicados.
n_sem_significado = df_profile.filter(F.col("significado").isNull()).count()

# Constrói um map literal do Spark e faz coalesce sobre o significado do dicionário.
mapa_manual = F.create_map(
    *[x for k, v in SIGNIFICADOS_MANUAIS.items() for x in (F.lit(k), F.lit(v))]
)

df_profile = (
    df_profile
    .withColumn(
        "origem_significado",
        F.when(F.col("significado").isNotNull(), "dicionário IBGE")
        .when(F.col("coluna").isin(list(SIGNIFICADOS_MANUAIS)), "inferido pelo grupo")
        .otherwise("não documentado"),
    )
    .withColumn("significado", F.coalesce(F.col("significado"), mapa_manual[F.col("coluna")]))
)

n_colunas = df_profile.count()
n_ainda_sem_significado = df_profile.filter(F.col("significado").isNull()).count()
n_dict_total = df_dict_lookup.count()
n_dict_orfao = (
    df_dict_lookup.join(df_profile.select("coluna"), on="coluna", how="left_anti").count()
)

print(f"Colunas perfiladas                     : {n_colunas}")
print(f"Colunas sem entrada no dicionário IBGE : {n_sem_significado}")
print(f"  das quais preenchidas manualmente     : {len(SIGNIFICADOS_MANUAIS)}")
print(f"  ainda sem documentação                 : {n_ainda_sem_significado}")
print(f"Códigos do dicionário sem coluna        : {n_dict_orfao} (de {n_dict_total})")

display(
    df_profile.filter(F.col("origem_significado") == "inferido pelo grupo")
    .select("tabela", "coluna", "significado", "dominio_amostra", "pct_nao_aplicavel")
)

In [0]:
# Esta é a tabela que vai para o documento Word. A única coluna que ainda precisa de
# trabalho manual é "significado", onde o dicionário não tinha entrada.
df_dicionario = (
    df_profile
    .select(
        F.col("coluna").alias("campo"),
        F.col("significado"),
        F.col("tipo_detectado").alias("tipo"),
        F.col("dominio_amostra").alias("dominio"),
        F.concat(
            F.col("pct_nao_resposta").cast("string"), F.lit("% não-resposta; "),
            F.col("pct_nao_aplicavel").cast("string"), F.lit("% não aplicável"),
        ).alias("qualidade"),
        "tabela",
        F.col("origem_significado").alias("origem_do_significado"),
    )
    .orderBy("tabela", "campo")
)

display(df_dicionario)

## Item 2.1 — Divergência do dicionário entre edições

O mesmo código de variável pode significar coisas diferentes em edições diferentes
da MUNIC: o IBGE reescreve perguntas, embute outro ano de referência e, às vezes,
aproveita códigos para perguntas novas. O dicionário da bronze é particionado por
edição justamente para permitir isso — cada ano guarda o seu próprio significado.

Esta seção compara o dicionário de **todas as edições carregadas** (não só a corrente),
**em todos os pares possíveis**: com duas edições há 1 par (ex.: 2019×2024), com três
há 3, com N haverá N(N−1)/2. Cada par de edições gera suas próprias linhas, listando
os códigos comuns àquele par cujo significado divergiu, classificados por tipo:

1. **reescrita editorial** — mesma pergunta e semântica, texto diferente;
2. **ano de referência embutido** — texto idêntico depois de remover os anos
   (ex.: MGOV04 pergunta sobre "2018" numa edição e "2023" na outra);
3. **reformulação parcial** — a pergunta mudou de escopo, mas o assunto é o mesmo;
4. **suspeita de reutilização do código** — textos com pouca semelhança: ou o código
   virou outra pergunta, ou a extração capturou um rótulo de resposta no lugar da
   pergunta. Estes exigem investigação antes de qualquer uso.

A classificação é **heurística** (similaridade de texto) e a seção é **descritiva**,
não veredito: divergência entre edições é esperada, e o objetivo é que quem escrever
a configuração do radar de uma edição saiba de antemão em quais códigos não pode
confiar. Saída: tabela `dq_dict_drift`, regravada inteira a cada execução — ela é
**entre edições** (colunas `edicao_a`/`edicao_b`) e cobre todos os anos de uma vez.

In [0]:
# Lê o dicionário COMPLETO da bronze — todas as edições carregadas, diferente do resto
# deste notebook, que trabalha sobre a edição corrente (reference_year). O join abaixo
# compara cada PAR de edições: 2 edições -> 1 par, 3 -> 3 pares, N -> N(N-1)/2.
# Cada par gera suas próprias linhas de divergência.
df_dict_full = (
    spark.table(bronze_tbl_dict)
    .select(
        F.upper(F.col("cod_variable")).alias("cod_variable"),
        F.col("ano_edicao"),
        F.trim(F.col("meaning")).alias("meaning"),
    )
)

edicoes = [r["ano_edicao"] for r in df_dict_full.groupBy("ano_edicao").count().orderBy("ano_edicao").collect()]
print(f"Edições no dicionário da bronze: {edicoes}")

df_dict_drift = None

if len(edicoes) < 2:
    print("Apenas uma edição carregada — a comparação de divergência precisa de pelo menos duas.")
else:
    # Self-join: cada par de edições (a < b) em que o código aparece nas duas
    a, b = df_dict_full.alias("a"), df_dict_full.alias("b")
    df_pares = (
        a.join(b,
               (F.col("a.cod_variable") == F.col("b.cod_variable")) &
               (F.col("a.ano_edicao") < F.col("b.ano_edicao")))
        .select(
            F.col("a.cod_variable").alias("cod_variable"),
            F.col("a.ano_edicao").alias("edicao_a"),
            F.col("b.ano_edicao").alias("edicao_b"),
            F.col("a.meaning").alias("significado_a"),
            F.col("b.meaning").alias("significado_b"),
        )
    )
    n_comuns = df_pares.count()

    # Divergentes = significados diferentes depois de caixa/acentos insensíveis via lower/trim
    df_divergentes = df_pares.filter(
        F.lower(F.trim("significado_a")) != F.lower(F.trim("significado_b"))
    )

    # A classificação usa similaridade de texto — poucas dezenas de linhas, cabe em pandas.
    import re as _re
    from difflib import SequenceMatcher as _SM

    def _sem_anos(texto):  # ignora os anos de 4 dígitos: reconhece "2018 vs 2023"
        return _re.sub(r"\d{4}", "", texto.lower()).strip()

    def _classificar(s1, s2):
        if _sem_anos(s1) == _sem_anos(s2):
            return "ano de referência embutido"
        similaridade = _SM(None, s1.lower(), s2.lower()).ratio()
        if similaridade >= 0.75:
            return "reescrita editorial"
        if similaridade < 0.40:
            return "suspeita de reutilização do código"
        return "reformulação parcial"

    pdf_drift = df_divergentes.toPandas()
    pdf_drift["tipo_divergencia"] = [
        _classificar(s1, s2)
        for s1, s2 in zip(pdf_drift["significado_a"], pdf_drift["significado_b"])
    ]
    pdf_drift = pdf_drift.sort_values(["tipo_divergencia", "cod_variable"]).reset_index(drop=True)
    df_dict_drift = spark.createDataFrame(pdf_drift)

    n_div = len(pdf_drift)
    print(f"\nCódigos comuns entre as edições    : {n_comuns}")
    print(f"Códigos com significado divergente : {n_div} ({n_div / n_comuns * 100:.0f}% dos comuns)")
    print("\nDivergências por tipo:")
    display(df_dict_drift.groupBy("tipo_divergencia").count().orderBy("tipo_divergencia"))
    display(df_dict_drift.orderBy("tipo_divergencia", "cod_variable"))

## Item 3 — Qualidade dos dados

Uma seção por dimensão. Cada uma produz um número que vai direto para o documento.

### 3.1 Completude

**O que mede:** das células que o questionário aplicava a cada município, quantas
foram de fato respondidas. São três números que somam 100%:

* **preenchido** — houve resposta (Sim/Não, categoria, texto, número);
* **não-resposta** — o município podia responder e não respondeu (não sabe,
  não informou, recusa, vazio);
* **não aplicável** — a pergunta não se aplicava (`-`, filtro de salto do
  questionário); não é falha do município e não entra na soma de não-resposta.

O bloco abaixo calcula primeiro o panorama global (todas as colunas somadas) e
depois as 10 piores colunas em não-resposta — porque a ausência se concentra em
blocos específicos de perguntas, e é isso que o documento precisa mostrar.

In [0]:
# Visão global: soma as contagens de TODAS as colunas do df_profile (uma linha por
# coluna) e converte em percentuais sobre o total de células avaliadas.
tot = df_profile.agg(
    F.sum("n_preenchido").alias("preenchido"),
    F.sum("n_nao_resposta").alias("nao_resposta"),
    F.sum("n_nao_aplicavel").alias("nao_aplicavel"),
    F.sum("n_registros").alias("total"),
).collect()[0]

# Percentuais globais — os três números que abrem a seção 3.1 do documento.
# "não aplicável" NÃO é falha do município: a pergunta não se aplicava (salto do
# questionário), então ele é reportado em separado, e não somado à não-resposta.
pct_global_preenchido = round(tot["preenchido"] / tot["total"] * 100, 2)
pct_global_nao_resposta = round(tot["nao_resposta"] / tot["total"] * 100, 2)
pct_global_nao_aplicavel = round(tot["nao_aplicavel"] / tot["total"] * 100, 2)

print(f"Preenchido    : {pct_global_preenchido}%")
print(f"Não-resposta  : {pct_global_nao_resposta}%")
print(f"Não aplicável : {pct_global_nao_aplicavel}%")

# As piores colunas: a média global esconde que a ausência é concentrada —
# algumas poucas colunas carregam quase toda a não-resposta do conjunto.
print("\n10 piores colunas por não-resposta:")
display(
    df_profile.select("tabela", "coluna", "significado", "pct_nao_resposta", "pct_nao_aplicavel")
    .orderBy(F.desc("pct_nao_resposta")).limit(10)
)

In [0]:
# Validade aqui = o domínio observado corresponde ao que a coluna diz ser?
# Colunas binárias devem conter apenas Sim/Não; qualquer coisa além é violação de domínio.
# Colunas cujo único conteúdo é ausência são marcadas como estruturalmente vazias.
df_validade = (
    df_profile
    .withColumn(
        "achado_validade",
        F.when(F.col("tipo_detectado") == "sem_resposta", "coluna sem nenhuma resposta válida")
        .when((F.col("tipo_detectado") == "categorico") & (F.col("n_distintos") == 1),
              "um único valor observado")
        .when(F.col("pct_nao_aplicavel") > 50, "maioria não aplicável (filtro de salto)")
        .otherwise(None),
    )
    .filter(F.col("achado_validade").isNotNull())
    .select("tabela", "coluna", "tipo_detectado", "n_distintos",
            "pct_nao_aplicavel", "achado_validade")
)

n_validade = df_validade.count()
print(f"Colunas com achado de validade: {n_validade}")
display(df_validade.orderBy(F.desc("pct_nao_aplicavel")))

In [0]:
# O .distinct() é obrigatório: o .select() sozinho não deduplica, e sem ele o
# check se resumiria a recontar as linhas e "passar" sempre.
n_info_distinct = df_raw_info.select("cod_munic").distinct().count()
n_gov_distinct = df_raw_gov.select("cod_munic").distinct().count()

print(f"info: {n_info:,} linhas / {n_info_distinct:,} cod_munic distintos")
print(f"gov : {n_gov:,} linhas / {n_gov_distinct:,} cod_munic distintos")

unicidade_ok = (n_info == n_info_distinct) and (n_gov == n_gov_distinct)
print(f"\nChave primária única nas duas tabelas: {unicidade_ok}")

In [0]:
# O bronze_to_silver pega os descritores municipais apenas da aba info, assumindo
# que a aba gov os repete de forma idêntica. Ninguém tinha verificado isso.
cols_comparar = ["uf", "populacao", "faixa_populacao", "regiao"]

df_desc_info = df_raw_info.select("cod_munic", *[F.col(c).alias(f"{c}_info") for c in cols_comparar])
df_desc_gov = df_raw_gov.select("cod_munic", *[F.col(c).alias(f"{c}_gov") for c in cols_comparar])

df_desc = df_desc_info.join(df_desc_gov, on="cod_munic", how="inner")

divergencias = {}
for c in cols_comparar:
    divergencias[c] = df_desc.filter(
        ~F.col(f"{c}_info").eqNullSafe(F.col(f"{c}_gov"))
    ).count()

for c, n in divergencias.items():
    print(f"{c:18s}: {n} municípios divergem entre as duas abas")

n_divergencias = sum(divergencias.values())

# Cobertura da chave: as duas abas descrevem exatamente os mesmos municípios?
n_so_info = df_raw_info.select("cod_munic").subtract(df_raw_gov.select("cod_munic")).count()
n_so_gov = df_raw_gov.select("cod_munic").subtract(df_raw_info.select("cod_munic")).count()
print(f"\nSó na info: {n_so_info} | Só na gov: {n_so_gov}")

In [0]:
# As colunas de linhagem gravadas pelo landing_to_bronze respondem esta dimensão diretamente.
df_lineage = df_raw_gov.select(
    *[c for c in COLS_METADADOS if c in df_raw_gov.columns]
).distinct()

display(df_lineage)

lineage = df_lineage.collect()[0].asDict() if df_lineage.count() == 1 else {}
ref_year = lineage.get("reference_year", "?")
ingestion = lineage.get("ingestion_timestamp", None)

dias_desde_ingestao = (
    spark.sql(f"SELECT datediff(current_date(), date('{ingestion}')) AS d").collect()[0]["d"]
    if ingestion else None
)
print(f"\nAno de referência: {ref_year} | dias desde a ingestão: {dias_desde_ingestao}")

### 3.6 Acurácia — detecção empírica de filtros de salto

Não há fonte externa para confrontar as respostas, então a aproximação possível é
a **coerência interna**: se a pergunta B só se aplica quando a pergunta A foi "Sim",
então "A = Não" deveria implicar "B = `-`".

A seção tem **dois blocos de código**, cada um responsável por uma coluna de
veredito diferente na `dq_skiplogic_candidates`:

* **o detector** (primeiro bloco) — propõe os pares pai/filha e emite
  **`veredito_dados`**, o veredito estatístico;
* **a verificação manual** (segundo bloco) — registra a conferência humana contra o
  questionário em **`verificacao_questionario`**.

### O detector — origem de `veredito_dados`

Para cada coluna com participação relevante de `-` (a "filha"), o detector procura
qual coluna-pai melhor explica o padrão de saltos — binária ou categórica curta, com
poucos valores distintos, porque o pai verdadeiro nem sempre é Sim/Não. Para cada par
proposto, ele registra as respostas que disparam o salto (`valores_gatilho`), a
cobertura e a precisão da relação, e o veredito estatístico: **`sim`** quando cobertura
e precisão ≥ 95%, **`parcial`** abaixo disso — o que costuma indicar variável cuja não
aplicabilidade decorre de uma cadeia de salto de mais de um nível, que este
procedimento não percorre. O resultado do detector é **hipótese**, não conclusão.

Três cuidados de desenho, cada um corrigindo um modo de falhar observado na base:

1. **O pai nem sempre é binário.** `Mgov09` tem três respostas e o bloco abaixo dele é
   pulado em duas. Restringir a busca a pais Sim/Não faz o detector descartar o pai
   verdadeiro e eleger uma coluna apenas correlacionada.
2. **O gatilho pode ser um conjunto de valores.** `Mgov33` é pulada quando `Mgov32` é
   "Não" **ou** "Não sabe informar". Testar um valor por vez só explica parte dos casos.
3. **O gatilho tem de ser uma resposta.** Se `-` pudesse disparar o salto, o detector
   elegeria um irmão do mesmo bloco — duas colunas que ficam vazias juntas por
   consequência compartilhada, não por causa. Códigos de não-resposta, ao contrário,
   são gatilhos legítimos.

Pares com cobertura ou precisão abaixo de 95% são marcados como `parcial`: não são
incoerências, e sim variáveis cuja não aplicabilidade decorre de uma cadeia de mais de
um nível, que este procedimento não percorre.

### A verificação manual — origem de `verificacao_questionario`

Nenhum detector que só olha os dados pode afirmar que o salto existe de fato no
questionário — essa conferência é humana. O segundo bloco aplica o dicionário
`VERIFICACOES_QUESTIONARIO`, editável direto na célula: cada entrada registra o
veredito do grupo para um par proposto pelo detector — `True` = o salto existe mesmo
no questionário; `False` = o detector elegeu uma correlação, não uma causa. O que não
estiver no dicionário permanece NULL = verificação pendente. O bloco imprime o resumo
(verificados × pendentes) e a **lista de trabalho**: os pares pendentes com pai,
gatilho, cobertura e precisão, ordenados da maior cobertura para a menor — é por ela
que a conferência contra o questionário da edição deve caminhar.

**Como as duas colunas se casam:** `veredito_dados` responde "os dados comportam essa
relação?" (detector); `verificacao_questionario` responde "o questionário confirma?"
(auditoria do grupo). As duas viajam juntas na `dq_skiplogic_candidates`: o gold usa
`veredito_dados = 'sim'` para montar o `grupo_pergunta`, e a verificação manual fica
registrada para o rigor de auditoria do relatório — sem ela, o documento não pode
afirmar que os saltos foram confirmados no questionário.

In [0]:
# Dois grupos de colunas entram na busca:
#   pais   -> colunas com poucos valores distintos (binárias OU categóricas curtas)
#   filhas -> colunas com participação relevante de "-"
#
# O pai NÃO precisa ser binário. Mgov09 ("estrutura de controle interno") tem três
# respostas, e o bloco abaixo dela é pulado em duas delas — uma busca só-binária
# perderia esse pai e elegeria uma coluna apenas correlacionada, sem relação de causa.
MIN_PCT_NAO_APLICAVEL = 5.0
MAX_VALORES_PAI = 8
MIN_CONFIRMACAO = 95.0   # cobertura e precisão, em %

# O gatilho precisa ser uma RESPOSTA de verdade. Se "-" pudesse disparar o salto, o
# detector elegeria um irmão do mesmo bloco — duas colunas que esvaziam juntas, o
# que é consequência compartilhada, não causa. Códigos de não-resposta PODEM ser
# gatilho: "Não sabe informar" pula legitimamente a pergunta seguinte.
NAO_PODE_SER_GATILHO = {"-", "None", "nan", ""}

pdf_prof = df_profile.select(
    "tabela", "coluna", "tipo_detectado", "pct_nao_aplicavel", "n_distintos", "significado",
).toPandas()

cols_filha = pdf_prof.loc[pdf_prof["pct_nao_aplicavel"] >= MIN_PCT_NAO_APLICAVEL, "coluna"].tolist()
cols_pai = pdf_prof.loc[pdf_prof["n_distintos"] <= MAX_VALORES_PAI, "coluna"].tolist()

print(f"Pais candidatos (<= {MAX_VALORES_PAI} valores distintos)   : {len(cols_pai)}")
print(f"Filhas candidatas (>= {MIN_PCT_NAO_APLICAVEL}% não aplicável): {len(cols_filha)}")

df_skip_candidates = None

if cols_pai and cols_filha:
    pdf_long = (
        df_long.filter(F.col("coluna").isin(list(set(cols_pai + cols_filha))))
        .select("cod_munic", "coluna", "valor")
        .toPandas()
    )
    mat = pdf_long.pivot(index="cod_munic", columns="coluna", values="valor").astype(str)

    # Y[i, j] = município i não aplicou a filha j
    Y = mat.reindex(columns=cols_filha).eq("-").to_numpy()
    n_y = Y.sum(axis=0)

    melhor_f1 = np.zeros(len(cols_filha))
    melhor = [None] * len(cols_filha)

    for pai in cols_pai:
        valores = sorted(mat[pai].unique())
        validos = np.array([v not in NAO_PODE_SER_GATILHO for v in valores])
        if not validos.any():
            continue

        # One-hot das respostas do pai: M[k, i] = município i respondeu valores[k]
        M = np.stack([(mat[pai] == v).to_numpy() for v in valores])
        tot = M.sum(axis=1)                                  # com que frequência cada resposta ocorre
        inter = M.astype(np.int32) @ Y.astype(np.int32)      # coocorrência resposta x filha

        with np.errstate(divide="ignore", invalid="ignore"):
            taxa = np.where(tot[:, None] > 0, inter / tot[:, None], 0.0)

        # O gatilho é o CONJUNTO de respostas sob as quais a filha fica majoritariamente "-".
        # Eleger um conjunto, em vez de um único valor, é o que captura saltos compostos
        # como "Não" OU "Não sabe informar".
        G = (taxa > 0.5) & validos[:, None]

        n_gatilho = (G * tot[:, None]).sum(axis=0)
        n_acerto = (G * inter).sum(axis=0)

        with np.errstate(divide="ignore", invalid="ignore"):
            cobertura = np.where(n_y > 0, n_acerto / n_y, 0.0)
            precisao = np.where(n_gatilho > 0, n_acerto / n_gatilho, 0.0)

        f1 = np.where((cobertura + precisao) > 0,
                      2 * cobertura * precisao / (cobertura + precisao), 0.0)

        for j in np.nonzero(f1 > melhor_f1)[0]:
            if cols_filha[j] == pai:
                continue
            melhor_f1[j] = f1[j]
            melhor[j] = {
                "coluna_filha": cols_filha[j],
                "coluna_pai": pai,
                "valores_gatilho": " | ".join(valores[k] for k in range(len(valores)) if G[k, j]),
                "pct_cobertura": round(float(cobertura[j]) * 100, 2),
                "pct_precisao": round(float(precisao[j]) * 100, 2),
                "n_nao_aplicavel": int(n_y[j]),
                "verificacao_questionario": None,   # preenchida na célula seguinte, após conferir o questionário
            }

    pdf_skip = pd.DataFrame([m for m in melhor if m])
    pdf_skip["veredito_dados"] = np.where(
        (pdf_skip.pct_cobertura >= MIN_CONFIRMACAO) & (pdf_skip.pct_precisao >= MIN_CONFIRMACAO),
        "sim", "parcial",
    )
    pdf_skip = pdf_skip.sort_values(
        ["pct_cobertura", "pct_precisao"], ascending=False
    ).reset_index(drop=True)

    n_conf = int((pdf_skip.veredito_dados == "sim").sum())
    n_exato = int(((pdf_skip.pct_cobertura == 100) & (pdf_skip.pct_precisao == 100)).sum())
    n_composto = int(pdf_skip.valores_gatilho.str.contains(r"\|").sum())

    print(f"\nFilhas avaliadas         : {len(pdf_skip)}")
    print(f"Veredito dos dados = sim : {n_conf} ({n_conf / len(pdf_skip) * 100:.0f}%)")
    print(f"  exatas (100/100)       : {n_exato}")
    print(f"Gatilhos compostos       : {n_composto}")
    print(f"Parciais (multi-nível)   : {len(pdf_skip) - n_conf}")

    df_skip_candidates = spark.createDataFrame(pdf_skip)
    display(df_skip_candidates)
else:
    print("Candidatos insuficientes para rodar o detector.")

In [0]:
# A coluna `verificacao_questionario` é a verificação HUMANA contra o questionário —
# nenhum detector que só olha os dados pode preenchê-la. Este é o passo manual da
# acurácia: abra o questionário da MUNIC desta edição, confira os pares propostos e
# registre o veredito por coluna-filha — True = o salto existe mesmo no questionário;
# False = o detector elegeu uma correlação, não uma causa. O que ficar de fora
# permanece NULL = verificação pendente. Mesmo espírito dos SIGNIFICADOS_MANUAIS
# da seção 2. (Já `veredito_dados` é o critério estatístico do detector — sim/parcial.)
VERIFICACOES_QUESTIONARIO = {
    # "mtic2541": True,    # exemplo — confirmado no questionário
    # "mtic186":  False,   # exemplo — correlação, não causa
}

if df_skip_candidates is not None:
    if VERIFICACOES_QUESTIONARIO:
        # Coalesce dos vereditos manuais: casa a coluna-filha e devolve True/False;
        # sem casamento, NULL, e o valor original (pendente) é preservado.
        vereditos = F.coalesce(*[
            F.when(F.col("coluna_filha") == filha, F.lit(v))
             .otherwise(F.lit(None).cast("boolean"))
            for filha, v in VERIFICACOES_QUESTIONARIO.items()
        ])
        df_skip_candidates = df_skip_candidates.withColumn(
            "verificacao_questionario",
            F.coalesce(vereditos, F.col("verificacao_questionario")),
        )

    n_conf = df_skip_candidates.filter(F.col("verificacao_questionario").isNotNull()).count()
    n_pend = df_skip_candidates.filter(F.col("verificacao_questionario").isNull()).count()
    print(f"Verificados contra o questionário : {n_conf}")
    print(f"Pendentes de conferência          : {n_pend}")

    # Lista de trabalho: pendentes ordenados pela cobertura — quanto mais alto o
    # par, mais barata a conferência e mais provável a confirmação. Preencha o
    # dicionário acima na próxima execução; os vereditos viajam na dq_skiplogic_candidates.
    display(
        df_skip_candidates
        .filter(F.col("verificacao_questionario").isNull())
        .select("coluna_filha", "coluna_pai", "valores_gatilho",
                "pct_cobertura", "pct_precisao", "veredito_dados")
        .orderBy(F.desc("pct_cobertura"), F.desc("pct_precisao"))
        .limit(15)
    )

## Viés e grupos sub-representados

Se a ausência de resposta estiver correlacionada com o porte do município, ela não
é aleatória — e tratá-la como se fosse enviesa toda a solução.

**Atenção à unidade de análise.** Medir a não-resposta sobre todas as colunas de uma vez
dilui o sinal: em quase todas elas a não-resposta é próxima de zero, e a média acaba
dizendo que o porte quase não importa. O viés que interessa está no bloco de LGPD,
onde está a variável-alvo do projeto. As duas medidas estão abaixo, nessa ordem:
a agregada como **contexto**, a do bloco LGPD como **evidência**.

In [0]:
# Contexto: a não-resposta medida sobre TODAS as colunas de uma vez. A leitura
# agregada dilui o sinal — por isso ela serve só de pano de fundo para a seção de viés.
df_faixa = df_raw_info.select("cod_munic", "faixa_populacao")

df_por_faixa = (
    df_long.join(df_faixa, on="cod_munic", how="inner")
    .groupBy("faixa_populacao")
    .agg(
        F.count("*").alias("n_celulas"),
        F.sum(F.when(F.col("classe").isin("nao_sabe", "nao_informou", "recusa", "nulo"), 1)
              .otherwise(0)).alias("n_nao_resposta"),
        F.countDistinct("cod_munic").alias("n_municipios"),
    )
    .withColumn("pct_nao_resposta",
                F.round(F.col("n_nao_resposta") / F.col("n_celulas") * 100, 2))
    .orderBy("faixa_populacao")
)

display(df_por_faixa)

In [0]:
# As perguntas de LGPD são o coração analítico deste projeto e, nas edições em que
# foram coletadas, contam entre as piores colunas do conjunto de dados. Medidas
# isoladamente, a relação com o porte do município é o oposto do que o agregado
# acima sugere.
#
# ATENÇÃO a edições em que o bloco não foi coletado: na MUNIC 2019 o dicionário não
# registra MGOV32–36 e as colunas chegam 100% vazias — nesse caso, taxas de 100% de
# não-resposta e 0% de "Sim" são ausência de coleta, não viés. A flag `lgpd_coletada`
# abaixo detecta isso, imprime o aviso e suprime as Figuras 1 e 2 na edição afetada.
COLS_LGPD = ["mgov32", "mgov33", "mgov34", "mgov35", "mgov36"]
COL_ALVO = "mgov32"

df_lgpd_faixa = (
    df_long.filter(F.col("coluna").isin(COLS_LGPD))
    .join(df_faixa, on="cod_munic", how="inner")
    .groupBy("faixa_populacao")
    .agg(
        F.count("*").alias("n_celulas"),
        F.sum(F.when(F.col("classe").isin("nao_sabe", "nao_informou", "recusa", "nulo"), 1)
              .otherwise(0)).alias("n_nao_resposta"),
    )
    .withColumn("pct_nao_resposta",
                F.round(F.col("n_nao_resposta") / F.col("n_celulas") * 100, 2))
)

# O mesmo corte só para a variável-alvo, mais a parcela que respondeu "Sim".
# Os dois juntos são o argumento inteiro: municípios pequenos respondem menos E
# respondem "Sim" menos — logo, as respostas ausentes não estão ausentes ao acaso.
df_alvo_faixa = (
    df_long.filter(F.col("coluna") == COL_ALVO)
    .join(df_faixa, on="cod_munic", how="inner")
    .groupBy("faixa_populacao")
    .agg(
        F.count("*").alias("n_municipios"),
        F.sum(F.when(F.col("classe") != "preenchido", 1).otherwise(0)).alias("n_sem_resposta_alvo"),
        F.sum(F.when(F.col("valor") == "Sim", 1).otherwise(0)).alias("n_sim"),
    )
    .withColumn("pct_sem_resposta_alvo",
                F.round(F.col("n_sem_resposta_alvo") / F.col("n_municipios") * 100, 2))
    .withColumn("pct_sim",
                F.round(F.col("n_sim") / F.col("n_municipios") * 100, 2))
)

df_vies_faixa = (
    df_lgpd_faixa.select("faixa_populacao", F.col("pct_nao_resposta").alias("pct_nr_bloco_lgpd"))
    .join(
        df_alvo_faixa.select("faixa_populacao", "n_municipios",
                             "pct_sem_resposta_alvo", "pct_sim"),
        on="faixa_populacao", how="inner",
    )
    .orderBy("faixa_populacao")
)

# O bloco foi coletado nesta edição? Se a variável-alvo está 100% sem resposta em
# TODAS as faixas, as perguntas não existiam no questionário (o dicionário da edição
# não registra os códigos) — ausência de coleta, não viés. O aviso lista as colunas
# de LGPD realmente vazias nesta edição, em vez de presumir quais são.
lgpd_coletada = not bool(
    df_vies_faixa.toPandas()["pct_sem_resposta_alvo"].eq(100).all()
)
if not lgpd_coletada:
    colunas_vazias = (
        df_long.filter(F.col("coluna").isin(COLS_LGPD))
        .groupBy("coluna")
        .agg(F.sum(F.when(F.col("classe") == "preenchido", 1).otherwise(0))
             .alias("n_preenchido"))
        .filter("n_preenchido = 0")
        .select("coluna")
        .toPandas()["coluna"].tolist()
    )
    print(f"AVISO: as colunas de LGPD ({', '.join(colunas_vazias)}) estão 100% vazias "
          f"na edição {reference_year}: o dicionário desta edição não as registra, ou "
          "seja, as perguntas não existiam no questionário. As taxas de não-resposta "
          "refletem ausência de coleta, não viés. As figuras desta seção serão "
          "suprimidas; a evidência de viés por porte vale para as edições que coletaram "
          "o bloco.")

display(df_vies_faixa)

In [0]:
# Sub-representação é sobre quantos municípios caem em cada faixa.
df_representacao = (
    df_raw_info.groupBy("faixa_populacao").agg(F.count("*").alias("n_municipios"))
    .withColumn("pct_do_total", F.round(F.col("n_municipios") / n_info * 100, 2))
    .orderBy("faixa_populacao")
)

display(df_representacao)

## Funil de seleção de variáveis

O radar do IMDM se apoia num recorte das colunas da pesquisa. Este funil substitui a
escolha subjetiva ("quais perguntas me parecem interessantes") por **critérios
explícitos, aplicados nesta ordem** — a primeira regra que casa define o destino da
coluna, e o motivo do descarte fica registrado:

1. **sem resposta válida** — nenhuma célula preenchida em nenhum município;
2. **não-resposta acima de `LIMIAR_NAO_RESPOSTA`** (padrão 50%) — a **não-resposta**
   engloba as quatro ausências que são falha de resposta: **"Não sabe informar" /
   "Não sabe"**, **"Não informou"**, **"Recusa"** e **célula vazia/NULL**. O **"-"
   fica fora**: pergunta não aplicável (filtro de salto do questionário) é ausência
   estrutural, não falha do município. Acima do corte, mais da metade dos municípios
   deixou a coluna sem resposta — restam poucos dados para distinguir qualquer coisa;
3. **concentração acima de `LIMIAR_CONCENTRACAO`** (padrão 97%) — entre as respostas
   **preenchidas**, a mais frequente passa do corte: a coluna é quase constante
   (ex.: praticamente todo mundo responde "Sim") e não diferencia municípios,
   mesmo 100% completa;
4. **texto livre** — respostas discursivas (URL, denominações de órgãos, descrições)
   não viram categoria e ficam fora do recorte quantitativo;
5. **mantida** — sobreviveu aos quatro filtros e entra no recorte analítico.

Os dois cortes são os widgets do topo do notebook. O `status_funil` fica gravado na
própria tabela de perfil (`dq_column_profile`) — a decisão viaja junto com os dados
e o documento pode citar quantas colunas cada critério derrubou.

**O funil não remove coluna de tabela nenhuma.** Este notebook é de perfilamento,
somente leitura sobre os dados: ele produz o **registro** do veredito por coluna.
Quem aplica o recorte de fato é o **gold**, na configuração do radar
(`silver_to_gold`), usando este perfil como evidência. A silver também não filtra:
ela limpa **todas** as colunas (ausências, tipos) — recortar é decisão analítica,
não de limpeza.

In [0]:
df_funil = (
    df_profile
    .withColumn(
        "status_funil",
        F.when(F.col("tipo_detectado") == "sem_resposta", "descartada: sem resposta válida")
        .when(F.col("pct_nao_resposta") > LIMIAR_NAO_RESPOSTA,
              F.concat(F.lit("descartada: não-resposta > "), F.lit(LIMIAR_NAO_RESPOSTA), F.lit("%")))
        .when(F.col("pct_concentracao") > LIMIAR_CONCENTRACAO,
              F.concat(F.lit("descartada: concentração > "), F.lit(LIMIAR_CONCENTRACAO), F.lit("%")))
        .when(F.col("tipo_detectado") == "texto_livre", "descartada: texto livre")
        .otherwise("mantida"),
    )
)

display(df_funil.groupBy("status_funil").count().orderBy(F.desc("count")))

n_mantidas = df_funil.filter(F.col("status_funil") == "mantida").count()
print(f"\n{n_mantidas} de {n_colunas} colunas sobrevivem ao funil")

# A tabela de perfil carrega o status do funil, para que a decisão viaje junto com os dados
df_profile = df_funil

In [0]:
# Uma linha por dimensão de qualidade: o que foi verificado, o número e o veredito.
# Esta é a tabela que você cola no documento Word.
linhas_resumo = [
    ("Completude",
     f"Percentual de células preenchidas, de não-resposta e de não aplicável sobre as {n_colunas} colunas",
     f"{pct_global_preenchido}% preenchido; {pct_global_nao_resposta}% não-resposta; "
     f"{pct_global_nao_aplicavel}% não aplicável",
     "Ausência relevante e concentrada em colunas específicas"),

    ("Validade",
     "Aderência dos valores ao domínio esperado e identificação de colunas sem resposta válida",
     f"{n_validade} colunas com achado de validade",
     "Requer tratamento documentado antes do uso"),

    ("Consistência",
     "Comparação dos descritores municipais entre as duas abas e cobertura do dicionário",
     f"{n_divergencias} divergências de descritores entre as abas; {n_sem_significado} colunas "
     f"sem entrada no dicionário do IBGE ({len(SIGNIFICADOS_MANUAIS)} reconstruídas pelo grupo, "
     f"{n_ainda_sem_significado} ainda sem definição); {n_dict_orfao} códigos do dicionário sem "
     f"coluna correspondente",
     "Divergência de nomenclatura entre documentação e dados"),

    ("Unicidade",
     "Contagem de valores distintos da chave cod_munic em cada aba",
     f"info: {n_info_distinct:,} de {n_info:,}; gov: {n_gov_distinct:,} de {n_gov:,}",
     "Chave íntegra" if unicidade_ok else "Duplicidade detectada"),

    ("Atualidade",
     "Defasagem entre ano de referência, data de ingestão e data de uso",
     f"ano de referência {ref_year}; {dias_desde_ingestao} dias desde a ingestão",
     "Dado anual; defasagem compatível com o uso pretendido"),

    ("Acurácia",
     "Coerência interna entre colunas encadeadas por filtro de salto do questionário",
     f"{0 if df_skip_candidates is None else df_skip_candidates.count()} pares pai/filha "
     f"candidatos identificados; confirmação pendente no questionário",
     "Aproximação por coerência interna; sem fonte externa de confronto"),
]

df_resumo = spark.createDataFrame(
    linhas_resumo, schema=["dimensao", "verificacao", "evidencia", "avaliacao"]
)

display(df_resumo)

## Gráficos (figuras do documento)

In [0]:
import matplotlib.pyplot as plt

if not lgpd_coletada:
    print("Figura 1 suprimida: o bloco de LGPD não foi coletado nesta edição — a figura "
          "não tem evidência real a mostrar aqui (veja o aviso na célula de viés).")
else:
    pdf_vies = df_vies_faixa.toPandas()

    # Série única: legenda desnecessária, o título já nomeia. Uma cor de destaque, eixos
    # discretos e rótulos diretos nas barras, em vez de um valor em cada linha da grade.
    ACCENT = "#2F6F9F"
    INK = "#3C3C3C"

    fig, ax = plt.subplots(figsize=(9, 4.5), dpi=150)
    barras = ax.bar(pdf_vies["faixa_populacao"], pdf_vies["pct_nr_bloco_lgpd"],
                    color=ACCENT, width=0.62)

    for b, v in zip(barras, pdf_vies["pct_nr_bloco_lgpd"]):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.25, f"{v:.1f}%",
                ha="center", va="bottom", fontsize=9, color=INK)

    ax.set_title("Não-resposta no bloco de LGPD, por faixa populacional",
                 fontsize=12, color=INK, pad=12)
    ax.set_ylabel("% de células sem resposta", fontsize=10, color=INK)
    ax.tick_params(axis="x", rotation=30, labelsize=8, colors=INK)
    ax.tick_params(axis="y", labelsize=9, colors=INK)
    ax.grid(axis="y", color="#E4E4E4", linewidth=0.8)
    ax.set_axisbelow(True)
    for lado in ["top", "right", "left"]:
        ax.spines[lado].set_visible(False)
    ax.spines["bottom"].set_color("#CCCCCC")
    plt.setp(ax.get_xticklabels(), ha="right")
    plt.tight_layout()
    plt.show()

In [0]:
# Segunda metade do argumento do viés: a própria resposta segue o porte do município.
# Suprimida nas edições em que o bloco de LGPD não foi coletado (ver flag `lgpd_coletada`
# na célula de viés) — sem o bloco, a figura não tem evidência a mostrar.
if not lgpd_coletada:
    print("Figura 2 suprimida: o bloco de LGPD não foi coletado nesta edição — a "
          "variável-alvo não tem respostas a mostrar aqui (veja o aviso na célula de "
          "viés).")
else:
    pdf_alvo = pdf_vies

    fig, ax = plt.subplots(figsize=(9, 4.5), dpi=150)
    barras = ax.bar(pdf_alvo["faixa_populacao"], pdf_alvo["pct_sim"],
                    color=ACCENT, width=0.62)

    for b, v in zip(barras, pdf_alvo["pct_sim"]):
        ax.text(b.get_x() + b.get_width() / 2, v + 1.2, f"{v:.1f}%",
                ha="center", va="bottom", fontsize=9, color=INK)

    ax.set_title("Municípios com área ou pessoa responsável pela LGPD (mgov32 = Sim)",
                 fontsize=12, color=INK, pad=12)
    ax.set_ylabel("% dos municípios da faixa", fontsize=10, color=INK)
    ax.tick_params(axis="x", rotation=30, labelsize=8, colors=INK)
    ax.tick_params(axis="y", labelsize=9, colors=INK)
    ax.grid(axis="y", color="#E4E4E4", linewidth=0.8)
    ax.set_axisbelow(True)
    for lado in ["top", "right", "left"]:
        ax.spines[lado].set_visible(False)
    ax.spines["bottom"].set_color("#CCCCCC")
    plt.setp(ax.get_xticklabels(), ha="right")
    plt.tight_layout()
    plt.show()

In [0]:
# Distribuição dos municípios pelas faixas — o contexto demográfico das figuras de viés.
pdf_rep = df_representacao.toPandas()

fig, ax = plt.subplots(figsize=(9, 4.5), dpi=150)
barras = ax.bar(pdf_rep["faixa_populacao"], pdf_rep["n_municipios"],
                color=ACCENT, width=0.62)

for b, v in zip(barras, pdf_rep["n_municipios"]):
    ax.text(b.get_x() + b.get_width() / 2, v + 12, f"{v:,}".replace(",", "."),
            ha="center", va="bottom", fontsize=9, color=INK)

ax.set_title("Municípios por faixa populacional", fontsize=12, color=INK, pad=12)
ax.set_ylabel("Número de municípios", fontsize=10, color=INK)
ax.tick_params(axis="x", rotation=30, labelsize=8, colors=INK)
ax.tick_params(axis="y", labelsize=9, colors=INK)
ax.grid(axis="y", color="#E4E4E4", linewidth=0.8)
ax.set_axisbelow(True)
for lado in ["top", "right", "left"]:
    ax.spines[lado].set_visible(False)
ax.spines["bottom"].set_color("#CCCCCC")
plt.setp(ax.get_xticklabels(), ha="right")
plt.tight_layout()
plt.show()

## Persistência

Só grava se o widget `persistir_tabelas` estiver em `sim`. Enquanto estiver em `nao`,
o notebook é somente leitura e pode rodar sem permissão de escrita no catálogo.

As tabelas **por edição** (`dq_column_profile`, `dq_dimension_summary`,
`dq_nonresponse_by_size`, `dq_skiplogic_candidates`) acumulam todas as edições já
perfiladas: cada execução grava apenas a edição corrente (`replaceWhere` sobre a
coluna `ano_edicao`), preservando as anteriores. A `dq_dict_drift` é a exceção —
cobre todas as edições de uma vez e é regravada inteira a cada execução.

**Efeito na gold (Etapa 3):** `dq_column_profile` e `dq_skiplogic_candidates`
são insumo direto do `silver_to_gold` — a `dim_indicador` nasce delas. Com
`persistir_tabelas = "nao"`, este notebook segue somente leitura, mas a gold
usará a evidência da última execução com `"sim"` da edição corrente; num
workspace novo, o `silver_to_gold` aborta (guard de pré-requisito) até existir
ao menos uma execução com `"sim"` para a edição. Antes de construir a gold de uma
edição, garanta uma execução deste notebook com `"sim"` para ela.

In [0]:
def gravar(df, nome_tabela, descricao):
    """Grava um artefato de qualidade como tabela Delta, registrando por que ele existe."""
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .option("userMetadata", descricao)
        .saveAsTable(nome_tabela)
    )
    print(f"Gravada: {nome_tabela}")


def gravar_por_edicao(df, nome_tabela, descricao, ano):
    """
    Grava o artefato de qualidade de UMA edição, preservando as outras já gravadas.

    A coluna ano_edicao é adicionada ao artefato e o replaceWhere troca apenas as
    linhas da edição corrente — as edições anteriores ficam intactas na mesma tabela.
    Tabelas antigas (criadas antes da coluna existir) são migradas: regravadas
    inteiras, uma única vez, já com a coluna.
    """
    df_edicao = df.withColumn("ano_edicao", F.lit(int(ano)))

    if not spark.catalog.tableExists(nome_tabela):
        # Primeira gravação: cria a tabela já com a coluna ano_edicao.
        (df_edicao.write.format("delta").mode("overwrite")
         .option("userMetadata", descricao).saveAsTable(nome_tabela))
        print(f"Criada (edição {ano}): {nome_tabela}")
        return

    colunas_atuais = {c.name for c in spark.table(nome_tabela).schema.fields}
    if "ano_edicao" not in colunas_atuais:
        # Migração única: a tabela antiga não tem a coluna — regrava inteira com ela.
        (df_edicao.write.format("delta").mode("overwrite")
         .option("overwriteSchema", "true")
         .option("userMetadata", descricao).saveAsTable(nome_tabela))
        print(f"Migrada (edição {ano}): {nome_tabela} — coluna ano_edicao adicionada")
        return

    # Caso normal: substitui só a edição corrente, preservando as demais.
    (df_edicao.write.format("delta").mode("overwrite")
     .option("replaceWhere", f"ano_edicao = {int(ano)}")
     .option("userMetadata", descricao).saveAsTable(nome_tabela))
    print(f"Gravada (edição {ano}): {nome_tabela}")


if PERSISTIR:
    # A edição corrente, defensivamente convertida para int (mesmo padrão do
    # landing_to_bronze), é o valor gravado na coluna ano_edicao de cada artefato.
    ano_edicao = int(reference_year) if isinstance(reference_year, (int, float, str)) else 2024

    # valores_distintos é um array usado só para derivar as outras colunas; ele
    # incharia a tabela sem acrescentar nada que o perfil já não afirme.
    gravar_por_edicao(df_profile.drop("valores_distintos"), dq_tbl_profile,
           f"Perfil de qualidade por coluna da MUNIC {ano_edicao} - tabelas bronze",
           ano_edicao)
    gravar_por_edicao(df_resumo, dq_tbl_summary,
           f"Sintese das seis dimensoes de qualidade - MUNIC {ano_edicao}",
           ano_edicao)
    gravar_por_edicao(df_vies_faixa, dq_tbl_bysize,
           f"Nao-resposta no bloco LGPD e taxa de Sim do alvo, por faixa - MUNIC {ano_edicao}",
           ano_edicao)
    if df_skip_candidates is not None:
        gravar_por_edicao(df_skip_candidates, dq_tbl_skip,
               f"Pares pai/filha candidatos a filtro de salto - MUNIC {ano_edicao}",
               ano_edicao)
    # A dq_dict_drift é a única ENTRE edições: cobre todos os anos de uma vez
    # (colunas edicao_a/edicao_b) e por isso é regravada inteira, sem ano_edicao.
    if df_dict_drift is not None:
        gravar(df_dict_drift, dq_tbl_drift,
               "Codigos comuns a duas edicoes cujo significado divergiu - classificados por tipo")

    # Ledger (pipeline_manifest): a linha 'profiling' desta edição. O contrato
    # é o shape COM DADOS da edição (mesma semântica de bronze/silver); o universo
    # perfilado (406 — schema união + descritores) fica em detalhes.colunas_perfiladas
    # (docs/arquitetura/pipeline_manifest.md, seção 4 — observação sobre a edição 2019).
    indicadores_com_dados = (
        df_long.filter(F.col("classe") != "nulo")
        .filter(F.col("coluna").rlike("^(mtic|mgov)"))
        .select("coluna").distinct().count()
    )
    gravar_linha_manifest(
        etapa="profiling",
        ano_edicao=ano_edicao,
        tabela_destino=f"{dq_tbl_profile};{dq_tbl_summary};{dq_tbl_bysize};{dq_tbl_skip}",
        indicadores=int(indicadores_com_dados),
        detalhes={
            "colunas_perfiladas": str(n_colunas),
            "funil_mantidas": str(n_mantidas),
            "saltos_candidatos": str(0 if df_skip_candidates is None else df_skip_candidates.count()),
            "drift": str(0 if df_dict_drift is None else df_dict_drift.count()),
            "faixas_vies": str(df_vies_faixa.count()),
            "lgpd": "coletada" if lgpd_coletada else "nao_coletada",
        },
    )
else:
    print("persistir_tabelas = nao — nada foi gravado.")
    print("Use o botão de download de cada saída display() para exportar CSV.")

In [0]:
# Escrever no volume exige WRITE VOLUME. Se falhar, as saídas display() acima ainda
# têm botão de download — o suficiente para levar os números ao documento.
if PERSISTIR:
    pasta_saida = f"{landing_folder_path}/quality_outputs"
    # CSVs por edição levam sufixo do ano, para não sobrescrever os de outra edição;
    # o de divergências é entre edições (cobre todos os anos) e fica sem sufixo.
    sufixo_ano = f"_{int(reference_year)}"
    try:
        dbutils.fs.mkdirs(pasta_saida)
        df_dicionario.toPandas().to_csv(f"{pasta_saida}/dicionario_esqueleto{sufixo_ano}.csv",
                                        index=False, encoding="utf-8-sig")
        df_profile.drop("valores_distintos").toPandas().to_csv(
            f"{pasta_saida}/qualidade_colunas{sufixo_ano}.csv", index=False, encoding="utf-8-sig")
        df_resumo.toPandas().to_csv(f"{pasta_saida}/qualidade_sintese{sufixo_ano}.csv",
                                    index=False, encoding="utf-8-sig")
        # Evidência por trás da seção 3.8 (gráficos de viés) e da seção 3.6 (filtros de salto).
        # Sem estes dois, o relatório cita números que nenhum anexo carrega.
        df_vies_faixa.toPandas().to_csv(f"{pasta_saida}/vies_por_faixa{sufixo_ano}.csv",
                                        index=False, encoding="utf-8-sig")
        if df_skip_candidates is not None:
            df_skip_candidates.toPandas().to_csv(f"{pasta_saida}/filtros_de_salto{sufixo_ano}.csv",
                                                 index=False, encoding="utf-8-sig")
        if df_dict_drift is not None:
            df_dict_drift.toPandas().to_csv(f"{pasta_saida}/dicionario_divergencias.csv",
                                            index=False, encoding="utf-8-sig")
        print(f"Arquivos CSV gravados em {pasta_saida}")
    except Exception as e:
        print(f"Falha ao exportar CSV ({type(e).__name__}): {e}")
        print("Use o botão de download das saídas display() como alternativa.")

In [0]:
print("Perfilamento concluído.")